In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install -q dagshub mlflow
import dagshub
dagshub.init(repo_owner='skupr23', repo_name='ml_assignment2', mlflow=True)

import mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 7.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 82.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=119b3b0c-d536-42d6-abb1-aae60b76fb27&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e0866463dcd165e2e7c414c949a2025a0b71ab9e3d8ede72d3446900e6107317




Output()

Accessing as skupr23

Initialized MLflow to track repo "skupr23/ml_assignment2"

Repository skupr23/ml_assignment2 initialized!

In [4]:

import warnings
import numpy as np
import pandas as pd

import mlflow.sklearn

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import roc_auc_score

train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
test_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")
test_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv")


train = train_transaction.merge(train_identity, on="TransactionID", how="left")
test = test_transaction.merge(test_identity, on="TransactionID", how="left")

In [5]:
y = train["isFraud"].astype(int)
X = train.drop(columns=["isFraud", "TransactionID"])
test_ids = test["TransactionID"]
X_test_raw = test.drop(columns=["TransactionID"])

In [6]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
        X["TransactionAmt_decimal"] = ((X["TransactionAmt"] % 1) * 1000).round()
        X["TransactionAmt_is_round"] = (X["TransactionAmt"] % 1 == 0).astype(int)

        X["Transaction_day"] = X["TransactionDT"] // 86400
        X["Transaction_hour"] = (X["TransactionDT"] // 3600) % 24
        X["Transaction_week"] = X["Transaction_day"] // 7

        if "P_emaildomain" in X.columns:
            X["P_email_provider"] = X["P_emaildomain"].astype(str).str.split(".").str[0]

        if "R_emaildomain" in X.columns:
            X["R_email_provider"] = X["R_emaildomain"].astype(str).str.split(".").str[0]

        if "P_emaildomain" in X.columns and "R_emaildomain" in X.columns:
            X["same_email_domain"] = (X["P_emaildomain"] == X["R_emaildomain"]).astype(int)

        return X

feature_engineer = FeatureEngineer()
X_fe = feature_engineer.fit_transform(X)
X_test_fe = feature_engineer.transform(X_test_raw)

common_columns = X_fe.columns.intersection(X_test_fe.columns)
X_fe = X_fe[common_columns]

selected = X_fe.columns[X_fe.isna().mean() <= 0.90]
X_selected = X_fe[selected]

num_cols = X_selected.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_selected.select_dtypes(exclude=np.number).columns.tolist()
cat_cols = [c for c in cat_cols if X_selected[c].nunique(dropna=True) <= 50][:40]

final_cols = num_cols + cat_cols

In [7]:
fraud_idx = y[y == 1].index
nonfraud_idx = y[y == 0].index

fraud_count = len(fraud_idx)

fraud_rows = X_selected.loc[fraud_idx]


nonfraud_rows = X_selected.loc[nonfraud_idx].sample(
    n=fraud_count * 4,
    random_state=42
)


X_reduced = pd.concat([fraud_rows, nonfraud_rows])
y_reduced = y.loc[X_reduced.index]


shuffled = np.random.permutation(len(X_reduced))

X_reduced = X_reduced.iloc[shuffled]
y_reduced = y_reduced.iloc[shuffled]

print(X_reduced.shape)
print(y_reduced.value_counts())

(103315, 401)
isFraud
0    82652
1    20663
Name: count, dtype: int64


In [8]:
with mlflow.start_run(run_name="LogisticRegression_Feature_Selection"):
    mlflow.log_param("max_missing_ratio", 0.90)
    mlflow.log_param("max_categorical_cardinality", 50)
    mlflow.log_param("selected_numeric_columns", len(num_cols))
    mlflow.log_param("selected_categorical_columns", len(cat_cols))
    mlflow.log_param("selected_total_columns", len(final_cols))

🏃 View run LogisticRegression_Feature_Selection at: https://dagshub.com/skupr23/ml_assignment2.mlflow/#/experiments/0/runs/a00818e75798463cb809b854fbd3e9a2
🧪 View experiment at: https://dagshub.com/skupr23/ml_assignment2.mlflow/#/experiments/0


In [9]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
!pip install -q xgboost
from xgboost import XGBClassifier


tree_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ]), cat_cols)
])

model = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessing", tree_preprocessor),

    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ))
])

In [10]:
with mlflow.start_run(run_name="RandomForest_Cross_Validation"):
    cv = TimeSeriesSplit(n_splits=3)

    scores = cross_val_score(
        model,
        X_reduced,
        y_reduced,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1
    )

    mlflow.log_metric("cv_auc_mean", float(scores.mean()))
    mlflow.log_metric("cv_auc_std", float(scores.std()))

    for i, score in enumerate(scores, 1):
        mlflow.log_metric(f"cv_auc_fold_{i}", float(score))

🏃 View run RandomForest_Cross_Validation at: https://dagshub.com/skupr23/ml_assignment2.mlflow/#/experiments/0/runs/a2537e5497b64cebbec7a6ac329635ed
🧪 View experiment at: https://dagshub.com/skupr23/ml_assignment2.mlflow/#/experiments/0


In [12]:
with mlflow.start_run(run_name="RandomForest_Final_Training") as run:
    model.fit(X_reduced, y_reduced)

    train_pred = model.predict_proba(X_reduced)[:, 1]
    train_auc = roc_auc_score(y_reduced, train_pred)

    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 12)
    mlflow.log_param("min_samples_split", 20)
    mlflow.log_param("min_samples_leaf", 10)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("train_auc", float(train_auc))

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name="RandomForest",
        input_example=X_reduced.head(5)
    )

2026/05/04 08:50:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 08:50:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can 

🏃 View run RandomForest_Final_Training at: https://dagshub.com/skupr23/ml_assignment2.mlflow/#/experiments/0/runs/a1e3f671699f44a78ae95866c7842771
🧪 View experiment at: https://dagshub.com/skupr23/ml_assignment2.mlflow/#/experiments/0
